***

Preparing Workspace

***

In [ ]:
# Packages
import pandas as pd
import numpy as np
import json
import requests
import os
from functools import reduce
from tqdm import tqdm
import functools as ft
import math
pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_code    = os.path.join(path_git, 'Python Code', 'BLS')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'BLS')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'BLS', 'config')

print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

# Base URL for API V2
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

# Set API key
exec(open(os.path.join(path_config, 'api_key.txt')).read())
dict_api[user]

In [ ]:
# Execute script to prepare API request inputs
estimate = 'BLS'
exec(open(os.path.join(path_code, 'Step 01 - Supplemental Scripts', 'Step 01a - Prepare API Request Inputs.py')).read())

# Import objects
df_indicators = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Indicators')
df_indicators = df_indicators[df_indicators['Indicator Name'] == indicator_name]

export_loc = df_indicators['Export Location'].values[0]
folder     = df_indicators['Folder'         ].values[0]

print('')
print('Export location file path: ' + export_loc)
print('Folder name:               ' + folder    )
print('')

***

Processing

***

In [ ]:
# Import queried BLS data
export_title = '_'.join([indicator_name, geography, 'BLS']) + '_raw.csv'

df_bls = pd.read_csv(os.path.join(path_raw, export_title))
df_bls.head()

In [ ]:
df_bls1 = df_bls.copy()

df_bls1 = pd.melt(df_bls1, id_vars = ['year', 'periodName'], var_name = 'seriesID', value_name = 'value')    
df_bls1['date_'] = df_bls1['year'].astype('str') + '-' + df_bls1['periodName'].astype('str')
df_bls1['date_'] = pd.to_datetime(df_bls1['date_'])

if survey in ['SM', 'CE']:
    df_bls1['value'] = df_bls1['value'].astype('float32').apply(lambda x: x*1000)

if survey in ['SM', 'LA', 'CE']:
    df_bls1 = df_bls1.merge(df_series_area, on = 'seriesID')

    if survey in ['SM', 'CE']:
        df_bls1 = df_bls1.groupby(['date_', 'area_text', 'area_code', 'Variable'], as_index = False).agg(value = ('value', 'sum'))
        df_bls1 = df_bls1.pivot_table(index = ['date_', 'area_code', 'area_text'], columns = 'Variable', values = 'value').reset_index()
        df_bls1 = pd.melt(df_bls1, id_vars = ['date_', 'area_code', 'area_text'], var_name = 'Variable', value_name = 'Value')
        df_bls1 = df_bls1.merge(df_industries[['Variable', 'industry_code']], on = 'Variable')
        df_bls1 = df_bls1.sort_values(['area_code', 'area_text', 'date_', 'industry_code'], ascending = [True, True, False, True])

    if survey == 'LA':
        df_bls1 = df_bls1.sort_values(['MSA_ID', 'area_text', 'date_'], ascending = [True, True, False])
        df_bls1 = df_bls1[['date_', 'MSA_ID', 'area_text', 'value']]
        df_bls1['date_'] = df_bls1['date_'].astype('str')


df_bls1 = df_bls1.reset_index(drop = True)
display(df_bls1.head())



if indicator_name == 'Jobs_1':
    if percentages == 'Yes':
            # Estimate proportions by groupings
            df_bls1['Percentage'] = 100*df_bls1['Value'] / df_bls1.groupby(['area_text', 'date_'])['Value'].transform('sum')
                
            # Reshape data to wide format
            df_bls2_pct = df_bls1.pivot_table(index = ['area_text', 'date_']
                                               , columns = 'Variable'
                                               , values = 'Percentage').reset_index()
            df_bls2_pct = df_bls2_pct.sort_values(['area_text', 'date_'], ascending = [True, False])
    
    df_bls1_all = df_bls1.groupby(['date_', 'area_text'], as_index = False)['Value'].agg(sum)
    df_bls1_all['Variable'] = 'All'
    df_bls1_all['Percentage'] = np.nan
    df_bls1_all = df_bls1_all.merge(df_bls1[['area_text', 'area_code']].drop_duplicates(), on = 'area_text', how = 'left')
    
    df_bls1_all = pd.concat([df_bls1, df_bls1_all])
    df_bls1_all = df_bls1_all.sort_values(['area_text', 'date_', 'Variable'], ascending = [True, False, True])
    df_bls1 = df_bls1_all.copy()
    df_bls1 = df_bls1.reset_index(drop = True)
    df_bls1['date_'] = df_bls1['date_'].astype('str')
    display(df_bls1.head())
    

if indicator_name == 'Jobs_2':
    if geography == 'MSA':
        df_bls_msa = df_bls1.copy()
        df_bls_msa = df_bls_msa.rename(columns = {'area_text':'MSA'})
        df_bls_msa = df_bls_msa[df_bls_msa['MSA'].str.contains('Sacramento|Yuba')]
        df_bls_msa = df_bls_msa.drop_duplicates()
        df_bls_mpo = df_bls_msa.copy()
        df_bls_mpo = df_bls_mpo.groupby(['date_', 'Variable', 'industry_code'], as_index = False).agg(value = ('Value', 'sum'))
        df_bls_mpo['MPO'] = 'SACOG'
        df_bls_mpo = df_bls_mpo.pivot_table(index = ['MPO', 'date_'], columns = 'Variable', values = 'value').reset_index()
        df_bls_mpo = pd.melt(df_bls_mpo, id_vars = ['MPO', 'date_'], var_name = 'Variable', value_name = 'Value')
        df_bls_mpo = df_bls_mpo.merge(df_series_area[['Variable', 'industry_code']], on = 'Variable')
    
        if percentages == 'Yes':
            # Estimate proportions by groupings
            df_bls_msa['Percentage'] = 100*df_bls_msa['Value'] / df_bls_msa[~df_bls_msa['Variable'].isin(['Total Nonfarm', 'Total Private', 'State Government', 'Local Government'])].groupby(['MSA', 'date_'])['Value'].transform('sum')
            df_bls_mpo['Percentage'] = 100*df_bls_mpo['Value'] / df_bls_mpo[~df_bls_mpo['Variable'].isin(['Total Nonfarm', 'Total Private', 'State Government', 'Local Government'])].groupby(['MPO', 'date_'])['Value'].transform('sum')
    
    
        df_bls_msa = df_bls_msa.drop_duplicates(['MSA', 'date_', 'Variable', 'Value'])
        df_bls_mpo = df_bls_mpo.drop_duplicates(['MPO', 'date_', 'Variable', 'Value'])
    
        df_bls_msa = df_bls_msa.sort_values(['MSA', 'date_', 'industry_code'], ascending = [True, False, True])
        df_bls_mpo = df_bls_mpo.sort_values(['MPO', 'date_', 'industry_code'], ascending = [True, False, True])
        
        df_bls_msa = df_bls_msa.drop(['area_code', 'industry_code'], axis = 1)
        df_bls_mpo = df_bls_mpo.drop([             'industry_code'], axis = 1)
    
        df_bls_msa['date_'] = df_bls_msa['date_'].astype('str')
        df_bls_mpo['date_'] = df_bls_mpo['date_'].astype('str')
    
        df_bls_msa = df_bls_msa.reset_index(drop = True)
        df_bls_mpo = df_bls_mpo.reset_index(drop = True)

        display(df_bls_msa.head(), df_bls_mpo.head())
        
    if geography == 'National':
        df_bls1 = df_bls1.set_index(['date_']).reset_index()
        df_bls1 = df_bls1.drop(['area_code', 'industry_code'], axis = 1)
        df_bls1 = df_bls1.drop_duplicates()
        df_bls1['date_'] = df_bls1['date_'].astype('str')

        df_bls1['Percentage'] = 100*df_bls1['Value'] / df_bls1[~df_bls1['Variable'].isin(['Total Nonfarm', 'Total Private', 'State Government', 'Local Government'])].groupby(['area_text', 'date_'])['Value'].transform('sum')
        
        display(df_bls1.head())

if indicator_name == 'Jobs_3':
    df_bls1_1 = df_bls1[df_bls1['Variable'].isin(['Total Private'  , 'Government'       ])]
    df_bls1_2 = df_bls1[df_bls1['Variable'].isin(['Goods Producing', 'Service-Providing'])]

    df_bls1_1['Percentage'] = 100*df_bls1_1['Value'] / df_bls1_1.groupby(['area_text', 'date_'])['Value'].transform('sum')
    df_bls1_2['Percentage'] = 100*df_bls1_2['Value'] / df_bls1_2.groupby(['area_text', 'date_'])['Value'].transform('sum')

    df_bls1_1_all = df_bls1_1.groupby(['date_', 'area_text'], as_index = False)['Value'].agg(sum)
    df_bls1_1_all['Variable'] = 'All'
    df_bls1_1_all['Percentage'] = np.nan
    df_bls1_1_all = df_bls1_1_all.merge(df_bls1[['area_text', 'area_code']].drop_duplicates(), on = 'area_text', how = 'left')
    df_bls1_1_all = pd.concat([df_bls1_1, df_bls1_1_all])
    df_bls1_1_all = df_bls1_1_all.sort_values(['area_text', 'date_', 'Variable'], ascending = [True, False, True])
    df_bls1_1 = df_bls1_1_all.copy()
    
    df_bls1_2_all = df_bls1_2.groupby(['date_', 'area_text'], as_index = False)['Value'].agg(sum)
    df_bls1_2_all['Variable'] = 'All'
    df_bls1_2_all['Percentage'] = np.nan
    df_bls1_2_all = df_bls1_2_all.merge(df_bls1[['area_text', 'area_code']].drop_duplicates(), on = 'area_text', how = 'left')
    df_bls1_2_all = pd.concat([df_bls1_2, df_bls1_2_all])
    df_bls1_2_all = df_bls1_2_all.sort_values(['area_text', 'date_', 'Variable'], ascending = [True, False, True])
    df_bls1_2 = df_bls1_2_all.copy()

    df_bls1_1 = df_bls1_1.drop(['industry_code'], axis = 1)
    df_bls1_2 = df_bls1_2.drop(['industry_code'], axis = 1)

    df_bls1_1 = df_bls1_1.reset_index(drop = True)
    df_bls1_2 = df_bls1_2.reset_index(drop = True)
    
    df_bls1_1['date_'] = df_bls1_1['date_'].astype('str')
    df_bls1_2['date_'] = df_bls1_2['date_'].astype('str')

    display(df_bls1_1.head(), df_bls1_2.head())


# 

In [ ]:
if indicator_name == 'Jobs_1':
    df_bls1 = df_bls1.rename(columns = {'area_text':'MSA', 'Variable':'Sector', 'Value':'Total Jobs'})
    display(df_bls1.head())

if indicator_name == 'Labor_2':
    df_bls1 = df_bls1.rename(columns = {'area_text':'MSA', 'value':'Unemployment Rate'})
    

if indicator_name == 'Jobs_2':
    if geography == 'MSA':
        df_bls_msa = df_bls_msa.rename(columns = {'Variable':'Sector', 'Value':'Total Jobs'})
        df_bls_mpo = df_bls_mpo.rename(columns = {'Variable':'Sector', 'Value':'Total Jobs'})
        display(df_bls_msa.head(), df_bls_mpo.head())
    if geography == 'National':
        df_bls1 = df_bls1.rename(columns = {'area_text':'Geography', 'Variable':'Sector', 'Value':'Total Jobs'})
        display(df_bls1.head())

if indicator_name == 'Jobs_3':
    df_bls1_1 = df_bls1_1.rename(columns = {'Variable':'Sector', 'Value':'Total Jobs'})
    df_bls1_2 = df_bls1_2.rename(columns = {'Variable':'Sector', 'Value':'Total Jobs'})
    if geography == 'MSA':
        df_bls1_1 = df_bls1_1.rename(columns = {'area_text':'MSA', 'area_code':'MSA ID'})
        df_bls1_2 = df_bls1_2.rename(columns = {'area_text':'MSA', 'area_code':'MSA ID'})
    display(df_bls1_1.head(), df_bls1_2.head())


***

Exporting

***

In [ ]:
# Create about documentation page for export
sample_type = 'LA'
# df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0, estimate)
# print("Visual representation of the output for:", indicator_name)
# display(df_about)

In [ ]:
path_yaml = os.path.join(path_config0, 'dict_about.yaml')
MOE_thresh = None

try:
    with open(path_yaml, 'r') as yaml_file:
        dict_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
except FileNotFoundError:
    print(f"Error: The file at {path_yaml} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")

if estimate is None:
    df_dicto = pd.DataFrame.from_dict(dict_about[survey][indicator_name]).T.reset_index().rename(columns = {'index': 'Metadata', 0: 'Description'})
else:
    df_dicto = pd.DataFrame.from_dict(dict_about[estimate][survey][indicator_name]).T.reset_index().rename(columns = {'index': 'Metadata', 0: 'Description'})

df_dicto.loc[df_dicto['Metadata'] == 'Last Updated', 'Description'] = date.today().strftime('%Y-%m-%d')
df_dicto.loc[df_dicto['Metadata'] == 'Year(s)'     , 'Description'] = f"{year_start}-{year_end}"
df_dicto.loc[df_dicto['Metadata'] == 'Geography'   , 'Description'] = geography
if MOE_thresh is not None:
    df_dicto.loc[df_dicto['Metadata'] == 'Margin of Error Limit', 'Description'] = MOE_thresh

# Split notes into rows, for visual clarity in about
# Find the row with 'Notes', then use that to take the information
# Separate based off of NewLines, make the rows with this
# Make a blank row past the first one. This way, we don't have to see notes as a cell like 7 times.
# Create a df from the new separated rows. Drop the old notes row
# Combine original with new rows
# Finally, we split the notes
def split_notes(df):
    notes_row = df[df['Metadata'] == 'Notes'].copy()
    notes = notes_row['Description'].values[0]
    
    lines = notes.split('\\n')
    new_rows = [{'Metadata': 'Notes' if i == 0 else '', 'Description': line} for i, line in enumerate(lines) if line]
    
    new_df = pd.DataFrame(new_rows)
    df_filtered = df[df['Metadata'] != 'Notes']
   
    notes_df = pd.concat([df_filtered, new_df], ignore_index=True)
    
    return notes_df


df_about = split_notes(df_dicto)

df_about

In [ ]:
if geography == 'MSA':
    # workbook_name = indicator_name + ' MSA BLS ' + survey + '_Chamber Study Mission.xlsx'
    workbook_name = indicator_name + ' MSA BLS ' + survey + '.xlsx'


if geography == 'National':
    # workbook_name = indicator_name + ' MSA BLS ' + survey + '_Chamber Study Mission.xlsx'
    workbook_name = indicator_name + ' MSA BLS ' + survey + '_Chamber Study Mission.xlsx'


In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, export_loc, indicator_name + ' ' + folder)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )


# Export to csv
# df_bls1.to_csv(os.path.join(path_out_csv, name_output_csv), index = False)

# Export to excel

if indicator_name == 'Jobs_1':
    if geography == 'MSA':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header = False)
            df_bls1 .to_excel(writer, index = False, sheet_name = 'MSA'                  )
    
    if geography == 'National':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_bls1.to_excel(writer, index = False, sheet_name = 'National')

if indicator_name == 'Jobs_2':
    if geography == 'MSA':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_bls_msa.to_excel(writer, index = False, sheet_name = 'MSA')
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_bls_mpo.to_excel(writer, index = False, sheet_name = 'MPO')
    if geography == 'National':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_bls1.to_excel(writer, index = False, sheet_name = 'National')

if indicator_name == 'Jobs_3':
    if geography == 'MSA':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about .to_excel(writer, index = False, sheet_name = 'About', header = False)
            df_bls1_1.to_excel(writer, index = False, sheet_name = 'Government and Private')
            df_bls1_2.to_excel(writer, index = False, sheet_name = 'Goods and Services'    )

if indicator_name == 'Labor_2':
    if geography == 'MSA':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header = False)
            df_bls1 .to_excel(writer, index = False, sheet_name = 'MSA'                  )

print('')
print("Successfully exported")